### 🧠 Guardrails

- ***Guardrails are safety mechanisms that control what goes into and comes out of an AI agent.***

- They sit around your agent pipeline and ensure the agent:
    * only processes safe, appropriate inputs
    * only performs approved actions
    * only returns validated, compliant outputs

- They are implemented as **middleware** that intercepts execution:
    - **Before** the agent starts (input guardrails)
    - **After** it completes (output guardrails)
    - **Around** model and tool calls

### Common Use Cases:
| Use Case | Example |
|---|---|
| PII leakage prevention | Redact emails/credit cards before logging |
| Prompt injection blocking | Detect adversarial inputs |
| Harmful content filtering | Block dangerous requests |
| Business rule enforcement | Require approval for financial ops |
| Output quality validation | Ensure response meets safety standards |


---

> 📌 ***Docs Reference:*** https://docs.langchain.com/oss/python/langchain/guardrails

---

### ***Approaches to Guardrails***

- ***Deterministic Guardrails***
    - Rule-based: regex, keyword matching, explicit checks.
    - ✅ Fast, predictable, cost-effective.
    - ❌ May miss nuanced violations.

- ***Model-Based Guardrails***
    - Uses LLMs/classifiers for semantic understanding.
    - ✅ Catches subtle/nuanced issues.
    - ❌ Slower and more expensive.

---

In [1]:
## Deterministic Guardials 
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""

    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("------- Deterministic Guardrail --------")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

------- Deterministic Guardrail --------
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [2]:
## Model Based Guardials 
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

def model_based_guardrail(text: str) -> str:
    """
    Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE.
    """

    prompt = f"""
    Is the following user input safe to process?
    Reply with only 'SAFE' or 'UNSAFE'.
    Input: {text}
    """
    result = llm.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("------- Model-Based Guardrail ------")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

------- Model-Based Guardrail ------
🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
🚫 UNSAFE: Explain how malware spreads


---
### ***Built-in Guardrail: PII Detection Middleware***

- LangChain provides built-in `PIIMiddleware` for detecting and handling **Personally Identifiable Information (PII)**.

- ***Supported PII Types:***

| Type | Example |
|---|---|
| `email` | user@example.com |
| `credit_card` | 5105-1051-0510-5100 |
| `ip` | 192.168.1.1 |
| `mac_address` | 00:1A:2B:3C:4D:5E |
| `url` | https://secret-site.com |

- ***Strategies:***

| Strategy | Result |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-1234` |
| `hash` | `a8f5f167...` |
| `block` | Raises an exception |

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [4]:
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# create agent with PIL Middleware 
agent = create_agent(
    model = llm,
    tools = [customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask CC
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API Keys 
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)
print("Agent with PII middleware created successfully..✅")

Agent with PII middleware created successfully..✅


In [5]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me..?"
    }]
})

print("------ Agent Response ------")
print(result["messages"][-1].content)

------ Agent Response ------
I’m sorry, but I can’t help with that.


In [6]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content



### ***Built-in Guardrail: Human-in-the-Loop Middleware***

- Pauses agent execution before sensitive operations and waits for human approval.

- ***Best for:***
    - Financial transactions
    - Sending emails to external parties
    - Deleting production data
    - Any operation with significant business impact

- ***Key requirement:*** 
    - ***A `checkpointer` for state persistence across interrupts.***

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool 
from langchain.agents.middleware import HumanInTheLoopMiddleware

## Define Tools 
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

In [8]:
##  Create agent with HITL middleware
hitl_agent = create_agent(
    model=llm,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created..✅")

Human-in-the-Loop agent created..✅


In [10]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("------ Agent paused — awaiting human approval -----")

------ Agent paused — awaiting human approval -----


In [11]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("------ Approved! Final response -------")
print(approved_result["messages"][-1].content)

------ Approved! Final response -------
✅ Email sent to **team@company.com** with the subject “Q4 Results Summary.”  
If you’d like to tweak the message or add any attachments, just let me know!


In [13]:
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

print("----- Agent paused — awaiting human approval -----")

----- Agent paused — awaiting human approval -----


In [14]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("----- Rejected..! Final response -----")
print(rejected_result["messages"][-1].content)

----- Rejected..! Final response -----
I’m sorry, but I can’t execute that deletion right now. If you’d like me to proceed, please confirm that you want to delete all records from the `users` table where `active = false`.


### ***Custom Guardrail: Before-Agent Hook(Input Filter)***

- Use `before_agent()` to validate or block requests **before any LLM processing begins**.

- ***Best for:***
    - Keyword/content filtering
    - Authentication checks
    - Rate limiting
    - Blocking specific categories of requests

In [16]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

# Create agent with content filter
filtered_agent = create_agent(
    model=llm,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created..!")

Content filter agent created..!


In [17]:
# Test 1: Safe request  should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
**Machine learning (ML)** is a branch of artificial intelligence that focuses on building systems that can learn from data, identify patterns, and make decisions or predictions with minimal human intervention.

### Core ideas

| Concept | What it means |
|---------|---------------|
| **Learning from data** | Instead of being explicitly programmed for every task, an ML model is trained on examples (input–output pairs). It discovers statistical regularities that generalize to new, unseen data. |
| **Model** | A mathematical function or algorithm that maps inputs to outputs. Common models include linear regression, decision trees, neural networks, support vector machines, etc. |
| **Training** | The process of adjusting the model’s internal parameters (weights, thresholds, etc.) so that its predictions match the training data as closely as possible. |
| **Evaluation** | Measuring how well the trained model performs on data it hasn’t seen before (validation/test se

In [18]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


### ***Custom Guardrail: After-Agent Hook (Output Safety)***

- Use `after_agent()` to validate the final agent response **before the user sees it**.

- ***Best for:***
    - Model-based safety evaluation of outputs
    - Compliance scanning (e.g. legal, medical, financial disclaimers)
    - Quality validation
    - Removing sensitive info that slipped through

In [20]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = llm

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=llm,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created..!")

Output safety agent created..!


In [21]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is Medicare?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
**Medicare** is the United States federal health‑insurance program that primarily serves:

| Who it covers | Typical age or condition | Key features |
|---------------|--------------------------|--------------|
| **Part A (Hospital Insurance)** | All U.S. citizens or permanent residents who are 65 + or younger people with certain disabilities | Covers inpatient hospital stays, skilled nursing facility care, hospice, and some home‑health services. Usually premium‑free for most people who paid Medicare taxes while working. |
| **Part B (Medical Insurance)** | Same as Part A | Covers doctor visits, outpatient care, preventive services, and medical supplies. Requires a monthly premium (based on income). |
| **Part C (Medicare Advantage)** | Any Medicare‑eligible person | Private‑sector plans that combine Parts A & B (and often D). May offer extra benefits (vision, dental, prescription drugs). Must meet Medicare’s minimum coverage standards. |
| **Part D (Prescription‑Drug Coverag

In [22]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "How to make bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

Response:
I’m sorry, but I can’t help with that.


### ***Layered / Combined Guardrails***

- Stack multiple guardrails in the `middleware=[]` array. They execute ***in order***, building layered protection.

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model=llm,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input

        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created..!")

🏭 Production-grade agent with 5-layer guardrails created..!


In [26]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "test_001"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "how to make a bomb?"}]},
    config=config
)


print(result["messages"][-1].content)

I’m sorry, but I can’t help with that.


### 📝 Summary

| Guardrail Type | Hook | When it Runs | Best For |
|---|---|---|---|
| PII Middleware | Input/Output | Around model calls | Data privacy, compliance |
| Human-in-the-Loop | Tool level | Before sensitive tools | High-stakes decisions |
| Content Filter | `before_agent` | Start of invocation | Blocking bad inputs early |
| Safety Validator | `after_agent` | End of invocation | Output quality/safety |
| Custom Logic | Any hook | Anywhere | Any business rule |

- ***Key Takeaways***
   - ***Guardrails = Middleware*** — implement them via the `middleware=[]` parameter in `create_agent()`.

   - ***Layer your guardrails*** — defense in depth is best practice.

   - ***Deterministic first, model-based second*** — use cheap rule-based checks early to avoid expensive LLM calls.

   - ***Human-in-the-Loop requires a checkpointer*** — use `InMemorySaver` for dev, persistent store for production.

   - ***Custom middleware*** gives you full control via `before_agent()` and `after_agent()` hooks.